# F5-TTS Voice Cloning — Colab

Generates scam-call audio for **Mirage fire drills** (Phase 3).

**Output contract:** a `.wav` file that `POST /drills/{id}/audio` can serve.

> Ethical use only: voice samples are for drills run by the consenting user.
> Never clone a voice to deceive a third party.

In [ ]:
# 1. Environment (T4/A100 runtime recommended)
!nvidia-smi -L
%cd /content
!git clone https://github.com/SWivid/F5-TTS.git
%cd F5-TTS
!pip install -q -e .

In [ ]:
# 2. Reference voice sample (10-20s, clean, mono 24kHz)
from pathlib import Path

REF_WAV = Path("/content/reference_voice.wav")  # <- upload your sample here
assert REF_WAV.exists(), "Upload a reference voice sample first"

PROMPT_TEXT = """Replace with the transcript of the reference sample."""

In [ ]:
# 3. Generate a scam-drill script
SCAM_SCRIPT = (
    "Namaste, this is the bank KYC desk. Your account will be blocked in ten "
    "minutes unless you confirm your one time password. Please do not tell "
    "anyone in your family, this is strictly confidential."
)

print(SCAM_SCRIPT)

In [ ]:
# 4. Synthesize
import subprocess

OUT = "/content/drill_audio.wav"

!f5-tts_infer-cli \
  --ref-text "{PROMPT_TEXT}" \
  --ref-audio {REF_WAV} \
  --gen-text "{SCAM_SCRIPT}" \
  --output {OUT}

print("wrote", OUT)

In [ ]:
# 5. Play + download
from IPython.display import Audio, display

display(Audio(OUT))
from google.colab import files

files.download(OUT)

## Wiring into Mirage (Phase 3)

1. Download the `.wav` into the drill asset store (Supabase Storage).
2. Register its path in `drills.audio_asset_url`.
3. The frontend `<audio>` element streams it during the drill run phase.

See `ml/README.md` for the full pipeline map.